In [1]:
import sys 
import os 

ls = os.path.abspath('../')
sys.path.append(ls)
from model.sampler import *

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_rays = 1400
rays_o = torch.rand(N_rays, 3, device=device)
rays_d = torch.randn(N_rays, 3, device=device)
rays_d = rays_d / torch.norm(rays_d, dim=-1, keepdim=True)
bounds = torch.stack([torch.ones(N_rays, device=device) * 2.0,
                      torch.ones(N_rays, device=device) * 6.0], dim=-1)


In [3]:
coarse = StratifiedSampler(N_samples=64, perturb=1.0).to(device)
pts_coarse, z_vals_coarse = coarse(rays_o, rays_d, bounds)
print(f"Coarse: {pts_coarse.shape}, {z_vals_coarse.shape}")


Coarse: torch.Size([1400, 64, 3]), torch.Size([1400, 64])


In [4]:
density_coarse = torch.rand(N_rays, 64, device=device) * 5.0
weights_coarse = compute_weights(density_coarse, z_vals_coarse, rays_d)
print(f"Weights sum: {weights_coarse.sum(-1).mean():.4f}")

Weights sum: 0.9999


In [5]:
fine = HierarchicalSampler(N_importance=128, perturb=0.0).to(device)
pts_fine, z_vals_fine, extras = fine(rays_o, rays_d, z_vals_coarse, weights_coarse)
print(f"Fine: {pts_fine.shape}, {z_vals_fine.shape}")

Fine: torch.Size([1400, 192, 3]), torch.Size([1400, 192])
